# Figure 4 — pupil dilation and the population anchoring state

Pupil radius and gaze deviation against the task-anchoring state, across all sessions with eye tracking.

Does the animal's eye behave differently on trials when the MEC population is
task-anchored versus non-anchored? Two measures, both derived from the
DeepLabCut pupil landmarks:

- **pupil radius** — mean distance from the 8 perimeter landmarks to their centroid (arousal/attention proxy)
- **gaze deviation** — how far the trial's gaze profile departs from the animal's own typical gaze at the same track positions

Both are **z-scored within session** before pooling, because camera distance,
zoom and focus differ between sessions and raw pixels are not comparable across
them.

### What changed from the single-session version

- Runs over **every VR session with eye tracking** (80 of 81), not just M25 D24.
  That session is in fact the *only* one with no eye channels in `clark2025/` —
  the old notebook worked because it read the older `COHORT12/` copy, which no
  longer exists on this machine. The original is kept as
  `figure4_pupil_anchoring_SINGLE_SESSION.ipynb.bak`.
- Anchoring labels are **recomputed**, not read from `all_sessions_rasters/*_cache.npz`.
  Those caches date from 2026-08-29, before trials were clipped to the ephys
  window, so they carry a block of forced non-anchored trials at the start of
  each session that is an alignment artefact.
- Pupil size comes from the **landmark geometry**, not the `eye_dilation` channel
  (see the note in the batch cell for why).
- A session inclusion criteria block is marked out below, currently permissive.


In [ ]:
import sys, os, json, glob, re, time, warnings
sys.path.insert(0, '/Users/harryclark/Documents/spatial-manifolds/src')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import pynapple as nap
from scipy.stats import mannwhitneyu, wilcoxon, spearmanr, rankdata
warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'Arial'

from spatial_manifolds.anchoring import trial_cluster_labels

# The VR loaders, trial clipping and task constants are defined once in
# lick_raster_by_trial_type.ipynb and exec'd in here, so this notebook cannot
# drift from the analysis the rest of the paper uses. They arrive via
# globals().update(), which is invisible to static analysis and to a reader, so
# the names this notebook actually depends on are listed and checked explicitly
# below -- a missing one then fails here with a clear message rather than as a
# NameError several cells later.
NB = ('/Users/harryclark/Documents/spatial-manifolds/scripts/figures/'
      'AnchorDynamics2026/lick_raster_by_trial_type.ipynb')
_g = {}
for _c in json.load(open(NB))['cells']:
    if _c['cell_type'] != 'code':
        continue
    _s = ''.join(_c['source'])
    if _s.startswith('MOUSE, DAY') or 'INV = pd.DataFrame' in _s:
        continue                      # skip the example-figure and inventory cells
    exec(compile(_s, '<lick_nb>', 'exec'), _g)

REQUIRED = ['SOURCE_PATH', 'CLS', 'TL', 'BS', 'NBIN', 'RZ', 'MIN_CELLS',
            'ANCH_COLOR', 'NONANCH_COLOR', 'vr_paths', 'clip_trials', 'load_vr']
_missing = [k for k in REQUIRED if k not in _g]
if _missing:
    raise RuntimeError(f'lick_raster_by_trial_type.ipynb did not define: {_missing}')
globals().update({k: _g[k] for k in REQUIRED})

ANCH, NONANCH = ANCH_COLOR, NONANCH_COLOR
FIG = '/Users/harryclark/Documents/spatial-manifolds/scripts/figures/AnchorDynamics2026'
DATA_DIR = '/Users/harryclark/Documents/spatial-manifolds/data/eye_anchoring'
TRIALS_CSV = f'{DATA_DIR}/eye_anchoring_trials.csv'
os.makedirs(DATA_DIR, exist_ok=True)

LM = ['n', 'ne', 'e', 'se', 's', 'sw', 'w', 'nw']   # DLC pupil perimeter landmarks
LIK_MIN = 0.5        # mean landmark likelihood below this = blink / tracking loss

print('setup ready:', ', '.join(REQUIRED))


## How the two measures are defined

Panel A uses real DeepLabCut coordinates (M22 D35), so the drawn pupil has the
actual landmark spacing rather than an invented one.

Two facts established directly from the data and worth recording:

- `eye_x`/`eye_y` **are** the mean of the 8 landmark coordinates — exactly
  (r = 1.00000, max abs difference 0.0000 px). So the centroid is not a separate
  estimate and can be recomputed from the landmarks.
- `eye_dilation` is **not** recoverable from the landmarks: it correlates with the
  landmark-derived mean diameter at only r = 0.56–0.86 across sessions, at roughly
  twice its magnitude. Whatever it is, its definition is not in the file, so the
  analysis below uses the landmark radius instead.

In [ ]:
"""Schematic: how pupil radius and gaze deviation are measured.

Coordinates in panel A are real DeepLabCut output (M22 D35, three frames chosen
as the 5th, 50th and 95th percentile of pupil radius), not invented, so the
drawn pupil has the aspect and landmark spacing of the actual tracking.
"""
import numpy as np, matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch
plt.rcParams['font.family'] = 'Arial'

LM = ['n', 'ne', 'e', 'se', 's', 'sw', 'w', 'nw']
FRAMES = {                      # real frames, (x[8], y[8]) in camera pixels
    'constricted': ([119.58, 123.56, 125.76, 122.88, 118.55, 113.63, 113.31, 115.37],
                    [78.15, 81.04, 85.56, 89.20, 90.73, 86.91, 82.30, 79.03]),
    'typical':     ([118.74, 123.28, 125.97, 124.06, 116.64, 108.85, 107.60, 112.05],
                    [74.99, 79.22, 85.40, 90.46, 93.09, 89.13, 81.51, 76.01]),
    'dilated':     ([127.81, 135.40, 138.91, 134.31, 122.18, 109.90, 109.05, 115.90],
                    [64.33, 70.59, 82.82, 91.56, 96.33, 88.92, 75.70, 66.24])}
C_LM, C_CEN, C_R = '#e8871e', '#c04744', '#3b6ea5'
ANCH, NONANCH = '#9d5391', '#5f9ea0'

fig = plt.figure(figsize=(10, 7.4))
gs = fig.add_gridspec(2, 3, height_ratios=[1, .82], hspace=.78, wspace=.32)

# ---- A: the tracked pupil, and what radius means ----------------------------
axA = fig.add_subplot(gs[0, 0])
x, y = np.array(FRAMES['typical'][0]), np.array(FRAMES['typical'][1])
cx, cy = x.mean(), y.mean()
r = np.hypot(x - cx, y - cy).mean()
axA.add_patch(Circle((cx, cy), r, facecolor='0.92', edgecolor='0.6', lw=1, ls='--', zorder=0))
for i, k in enumerate(LM):
    axA.plot([cx, x[i]], [cy, y[i]], color=C_R, lw=.7, alpha=.55, zorder=1)
    axA.scatter(x[i], y[i], s=42, color=C_LM, zorder=3, edgecolor='w', linewidth=.8)
    off = 1.28 * (np.array([x[i], y[i]]) - np.array([cx, cy]))
    axA.text(cx + off[0], cy + off[1], k, fontsize=7, ha='center', va='center', color='0.35')
axA.scatter([cx], [cy], s=70, marker='+', color=C_CEN, zorder=4, linewidth=2)
axA.text(cx + 1.2, cy - 2.6, '(eye_x, eye_y)', fontsize=7.5, color=C_CEN)
axA.set_aspect('equal'); axA.invert_yaxis()      # image coordinates
axA.margins(.16)                                 # room for the landmark labels
axA.set_title('A  8 DeepLabCut pupil landmarks', fontsize=9.5, loc='left')
axA.set_xlabel('camera x (px)', fontsize=8); axA.set_ylabel('camera y (px)', fontsize=8)
axA.tick_params(labelsize=7)
axA.text(.0, -.22, 'centroid = mean of the 8 landmarks (exact)\n'
                   r'radius $r=\frac{1}{8}\sum_i \|\mathbf{p}_i-\mathbf{c}\|$',
         transform=axA.transAxes, fontsize=7.5, va='top', color='0.25')

# ---- B: radius tracks pupil size -------------------------------------------
axB = fig.add_subplot(gs[0, 1])
for j, (nm, (xx, yy)) in enumerate(FRAMES.items()):
    xx, yy = np.array(xx), np.array(yy)
    ccx, ccy = xx.mean(), yy.mean()
    rr = np.hypot(xx - ccx, yy - ccy).mean()
    dx = j * 34
    axB.add_patch(Circle((ccx - ccx + dx, 0), rr, facecolor=C_R, alpha=.16,
                         edgecolor=C_R, lw=1.2))
    axB.scatter(xx - ccx + dx, yy - ccy, s=20, color=C_LM, zorder=3,
                edgecolor='w', linewidth=.6)
    axB.scatter([dx], [0], s=44, marker='+', color=C_CEN, zorder=4, linewidth=1.6)
    axB.text(dx, 20, f'{nm}\nr = {rr:.1f} px', ha='center', va='top', fontsize=7.5, color='0.25')
axB.set_xlim(-24, 92); axB.set_ylim(-20, 34); axB.set_aspect('equal'); axB.axis('off')
axB.invert_yaxis()
axB.set_title('B  radius across the session', fontsize=9.5, loc='left')

# ---- C: z-scoring ----------------------------------------------------------
axC = fig.add_subplot(gs[0, 2])
rng = np.random.default_rng(0)
s1 = rng.normal(9.2, 1.3, 4000); s2 = rng.normal(21.5, 3.4, 4000)
for v, c, lab in ((s1, '#3b6ea5', 'session 1'), (s2, '#8a4fa8', 'session 2')):
    axC.hist(v, bins=50, density=True, color=c, alpha=.5, linewidth=0, label=lab)
axC.set_xlabel('raw radius (px)', fontsize=8); axC.set_ylabel('density', fontsize=8)
axC.tick_params(labelsize=7); axC.legend(fontsize=7, frameon=False)
axC.set_title('C  why z-score within session', fontsize=9.5, loc='left')
axC.text(.0, -.22, 'camera distance, zoom and focus differ per session, so raw\n'
                   'pixels are not comparable across them. Each measure is\n'
                   'z-scored within session, over valid trials only.',
         transform=axC.transAxes, fontsize=7.5, va='top', color='0.25')
for s in (axC.spines['top'], axC.spines['right']):
    s.set_visible(False)

# ---- D: gaze deviation ------------------------------------------------------
axD = fig.add_subplot(gs[1, :2])
nb = 100
pos = np.arange(nb) * 2
ref = 116 + 2.2 * np.sin(2 * np.pi * pos / 200) - 1.1 * np.cos(4 * np.pi * pos / 200)
tr = ref + rng.normal(0, .35, nb)
tr[38:62] += np.linspace(0, 5.5, 24) * np.hanning(24) * 1.6     # a deviating stretch
axD.plot(pos, ref, color='0.35', lw=2, label='reference: median gaze at each position bin')
axD.plot(pos, tr, color=ANCH, lw=1.4, label='one trial')
axD.fill_between(pos, ref, tr, color=ANCH, alpha=.22, linewidth=0, edgecolor='none')
axD.axvspan(90, 110, color='#9ecae1', alpha=.35, lw=0, zorder=0)
axD.text(100, .02, 'RZ', ha='center', va='bottom', fontsize=7, color='0.4',
         transform=axD.get_xaxis_transform())
axD.set_xlabel('Position on track (cm)', fontsize=8.5)
axD.set_ylabel('gaze (px)', fontsize=8.5)
axD.tick_params(labelsize=7); axD.legend(fontsize=7, frameon=False, loc='upper left')
axD.set_title('D  gaze deviation: each trial against the animal’s own typical gaze, bin by bin',
              fontsize=9.5, loc='left')
for s in (axD.spines['top'], axD.spines['right']):
    s.set_visible(False)
axD.text(.0, -.26, r'deviation$_t=\frac{1}{n_{bins}}\sum_b \sqrt{(x_{t,b}-\bar{x}_b)^2+(y_{t,b}-\bar{y}_b)^2}$'
                   '   — each trial against the session median at the SAME track position, so a whole-session\n'
                   'calibration drift (which shifts every bin together) cannot masquerade as a gaze shift.',
         transform=axD.transAxes, fontsize=7.5, va='top', color='0.25')

# ---- E: what gets compared --------------------------------------------------
axE = fig.add_subplot(gs[1, 2])
axE.axis('off')
axE.set_title('E  per trial, two z-scored measures', fontsize=9.5, loc='left')
lab = np.array([1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1])
for i, v in enumerate(lab):
    axE.add_patch(plt.Rectangle((i * .062, .70), .056, .16,
                                color=ANCH if v else NONANCH, lw=0))
axE.text(0, .90, 'population anchoring state (per trial)', fontsize=7.5, color='0.25')
axE.text(0, .60, 'z(pupil radius)\nz(gaze deviation)', fontsize=8, color='0.2', va='top')
axE.annotate('', xy=(.47, .50), xytext=(.47, .66),
             arrowprops=dict(arrowstyle='-|>', color='0.5', lw=1.1))
axE.text(0, .34, 'compared anchored vs non-anchored,\nwithin session, then across sessions',
         fontsize=7.5, color='0.25', va='top')
axE.set_xlim(0, 1); axE.set_ylim(0, 1)

_p = f'{FIG}/eye_measurement_schematic.pdf'
plt.savefig(_p, dpi=200, bbox_inches='tight')
print('saved', _p)

plt.show()

## Per-session extraction

~10 s per session, ~13 min for all 81. Writes one row per trial to
`data/eye_anchoring/eye_anchoring_trials.csv`, so the summary below can be
re-run without recomputing.

### Head-movement artifact removal

The animal is head-fixed, but the head can still shift in the headplate, which
translates the whole eye in the camera frame and displaces the measured gaze
with no eye movement at all. Transient tracking failures look different from
sustained slips and are handled separately — see the module docstring.

The step threshold is **calibrated, not assumed**, and the obvious choice fails:
a session's entire gaze excursion spans only ~4–10 px, about one pupil radius,
so a half-radius threshold sits inside the range of ordinary eye movement and
the correction then removes real gaze. The docstring records what that looked
like.

In [ ]:
"""Detect head-movement artifacts in the pupil centroid trace.

Two artifacts, needing different treatment:

TRANSIENT SPIKES -- the centroid jumps and comes back within a few frames.
Tracking glitches, partial occlusion by a whisker or the lid. Dropped outright.

SUSTAINED STEPS -- the centroid jumps and STAYS at the new baseline: the head has
shifted in the headplate, so the whole eye translates in the camera frame. Two
signatures are required together, and both matter:

  1. the displacement happens in a single frame (a real head slip is fast), and
  2. the baseline 2 s after differs from the baseline 2 s before.

Requiring (1) is what makes this work. An earlier version tested only (2), which
also fires continuously during smooth gaze drift; accumulating a correction from
those false events inflated one session's gaze range from 8 px to 219 px.

The pupil RADIUS must also be unchanged across the step, which is what separates a
head slip (the eye translates) from a genuine gaze shift with dilation.

A step cannot just be dropped -- everything after it is referenced to a baseline
that no longer applies -- so the cumulative step offset is subtracted, removing
the staircase while leaving real drift intact.

Thresholds scale with each session's own median pupil radius, because camera
distance and zoom differ between sessions (median radius 7.8-12.7 px here), so a
fixed pixel threshold is a different physical displacement in each.

STEP_R was calibrated, not assumed. The whole gaze excursion of a session spans
only ~4-10 px (2.5-97.5 pct), i.e. about one pupil radius, so a threshold of half
a radius sits inside the range of ordinary eye movement: at STEP_R=0.5 the
detector fired 0-29 times per session and the accumulated correction INFLATED one
session's gaze range from 9.7 px to 72 px, which is proof the events were real
gaze rather than slips. At STEP_R=1.0 it fires 0-6 times, removes <0.8% of frames,
and the corrected range matches the raw range (61.4 -> 61.2 px summed over nine
sessions) -- the signature of removing a few genuine discontinuities and nothing
else. A displacement of a full pupil radius within one 33 ms frame is not an eye
movement.
"""
import numpy as np, pandas as pd

SPIKE_R  = 0.8      # transient jump, in units of median pupil radius
STEP_R   = 1.0      # single-frame jump that must also shift the baseline
STEP_WIN = 2.0      # seconds each side over which the new baseline must hold
PAD_S    = 0.5      # seconds blanked either side of a detected step


def head_movement(cx, cy, rad, fps=30.0):
    """Returns (bad, cx_corr, cy_corr, n_spike, n_step)."""
    cx = np.asarray(cx, float).copy(); cy = np.asarray(cy, float).copy()
    rad = np.asarray(rad, float)
    n = len(cx)
    r_med = np.nanmedian(rad)
    if not np.isfinite(r_med) or r_med <= 0:
        return np.ones(n, bool), cx, cy, 0, 0
    t_spike, t_step = SPIKE_R * r_med, STEP_R * r_med
    w = max(3, int(STEP_WIN * fps))
    pad = max(1, int(PAD_S * fps))

    d = np.r_[0.0, np.hypot(np.diff(cx), np.diff(cy))]
    jump = np.where(d > t_step)[0]          # candidate discontinuities

    def med(a, lo, hi):
        s = a[max(0, lo):min(n, hi)]
        return np.nanmedian(s) if np.isfinite(s).sum() >= 3 else np.nan

    bad = np.zeros(n, bool)
    ox = np.zeros(n); oy = np.zeros(n)
    n_step = 0
    last = -10 ** 9
    for i in jump:
        if i - last < w:                    # at most one event per window
            continue
        px, py = med(cx, i - w, i), med(cy, i - w, i)
        qx, qy = med(cx, i + 1, i + 1 + w), med(cy, i + 1, i + 1 + w)
        pr, qr = med(rad, i - w, i), med(rad, i + 1, i + 1 + w)
        if not all(np.isfinite(v) for v in (px, py, qx, qy, pr, qr)):
            continue
        if np.hypot(qx - px, qy - py) > t_step and abs(qr - pr) / r_med < 0.15:
            ox[i + 1:] += qx - px
            oy[i + 1:] += qy - py
            bad[max(0, i - pad):min(n, i + pad + 1)] = True
            n_step += 1
            last = i

    cxc, cyc = cx - ox, cy - oy

    # transient spikes, judged on the CORRECTED trace so a step is not re-counted
    base_x = pd.Series(cxc).rolling(w, center=True, min_periods=w // 3).median().values
    base_y = pd.Series(cyc).rolling(w, center=True, min_periods=w // 3).median().values
    spike = np.hypot(cxc - base_x, cyc - base_y) > t_spike
    bad |= spike
    return bad, cxc, cyc, int(spike.sum()), n_step
print("head-movement detection ready")

### Per-session extraction

~11 s per session, ~15 min for all 81. Writes one row per trial to
`data/eye_anchoring/eye_anchoring_trials.csv`.

In [ ]:
"""Per-trial pupil radius and gaze deviation against population anchoring state,
for every VR session with eye tracking.

Anchoring labels are RECOMPUTED here rather than read from
`all_sessions_rasters/*_cache.npz`: those caches were written on 2026-08-29,
before trials were clipped to the ephys window, so they carry the block of
forced non-anchored trials at the start of each session that clipping removed.

Pupil size is measured as the mean distance from the 8 DeepLabCut perimeter
landmarks to their centroid, computed here, NOT from the `eye_dilation` channel
in the NWB. eye_dilation correlates with the landmark-derived diameter at only
r = 0.56-0.86 across sessions and is ~2x its magnitude, so it is some other
derived quantity whose definition is not recoverable from the file. The
landmark radius is transparent and reproducible; eye_dilation is not.
"""

# Imports, paths, LM, LIK_MIN, and the VR loaders all come from the setup cell.

def eye_channels(beh):
    return all(f'eye_{k}_{c}' in beh.keys() for k in LM for c in ('x', 'y', 'likelihood'))


def session_eye_trials(beh, n_tr):
    """Bin gaze centroid and pupil radius onto the trial x position-bin grid.

    Uses the SAME cumulative-travel feature, moving epoch, bin count and bounds as
    population_anchoring's tuning curves, so column b of these arrays is the same
    2 cm of track as column b of the neural rate maps.
    """
    tn, trav = beh['trial_number'], beh['travel']
    dt = trav - (float(np.asarray(tn.values)[0]) - 1) * TL
    moving = beh['S'].threshold(3.0, method='above').time_support

    lik = np.nanmean([np.asarray(beh[f'eye_{k}_likelihood'].values) for k in LM], axis=0)
    bad = lik < LIK_MIN
    X = np.array([np.asarray(beh[f'eye_{k}_x'].values, float) for k in LM])
    Y = np.array([np.asarray(beh[f'eye_{k}_y'].values, float) for k in LM])
    cx, cy = X.mean(axis=0), Y.mean(axis=0)           # == eye_x/eye_y exactly
    rad = np.hypot(X - cx, Y - cy).mean(axis=0)
    t = np.asarray(beh['eye_x'].index)

    # head-movement artifacts: drop transient spikes, and remove the staircase
    # left by sustained slips so post-slip trials are not all scored as deviant
    cxm, cym = cx.copy(), cy.copy(); cxm[bad] = np.nan; cym[bad] = np.nan
    radm = rad.copy(); radm[bad] = np.nan
    fps = 1.0 / np.median(np.diff(t))
    hm_bad, cx, cy, n_spike, n_step = head_movement(cxm, cym, radm, fps=fps)
    bad = bad | hm_bad

    def binned(v):
        v = v.copy(); v[bad] = np.nan
        tc = np.asarray(nap.compute_1d_tuning_curves_continuous(
            nap.Tsd(t=t, d=v), dt, nb_bins=n_tr * NBIN,
            minmax=[0, n_tr * TL], ep=moving)[0])
        # pynapple bug: occupancy comes from the dense ~60 Hz position trace but the
        # average from the sparse ~30 Hz eye camera, so a visited bin containing no
        # camera frame keeps the 0.0 placeholder instead of becoming NaN. A true
        # pixel coordinate of exactly 0 is impossible here (raw values never < ~90).
        tc[tc == 0.0] = np.nan
        return tc.reshape(n_tr, NBIN)

    return (binned(cx), binned(cy), binned(rad), float(np.mean(bad)),
            n_spike, n_step, float(np.mean(hm_bad)))


def anchoring_and_eye(mouse, day):
    bp, cp = vr_paths(mouse, day)
    beh = nap.load_file(bp); clusters = nap.load_file(cp)
    if not eye_channels(beh):
        return None
    trials_all = beh['trials'].as_dataframe()
    trials, orig = clip_trials(trials_all, clusters)
    keep = np.isin(trials_all.number.values.astype(int), orig)
    n_tr = len(trials_all)

    ids = CLS[(CLS.mouse == mouse) & (CLS.day == day) &
              (CLS.cell_class_of1.isin(['GC', 'NGS']))].cluster_id.astype(int)
    ids = [c for c in ids if c in clusters.index]
    if len(ids) < MIN_CELLS:
        return None
    tn, trav = beh['trial_number'], beh['travel']
    dt = trav - (float(np.asarray(tn.values)[0]) - 1) * TL
    moving = beh['S'].threshold(3.0, method='above').time_support
    tc = nap.compute_1d_tuning_curves(clusters[ids], dt, nb_bins=n_tr * NBIN,
                                      minmax=[0, n_tr * TL], ep=moving)
    labs = []
    for c in ids:
        M = np.asarray(tc[c]).reshape(n_tr, NBIN)[keep]
        lab, _, _, _ = trial_cluster_labels(M)
        if not np.isnan(lab).all():
            labs.append(lab)
    if not labs:
        return None
    L = np.array(labs)
    frac = np.nanmean(L, axis=0)

    ex, ey, er, blink, n_spike, n_step, hm_frac = session_eye_trials(beh, n_tr)
    ex, ey, er = ex[keep], ey[keep], er[keep]

    # gaze deviation: distance from the session's median gaze AT THE SAME position bin
    ref_x, ref_y = np.nanmedian(ex, axis=0), np.nanmedian(ey, axis=0)
    dev = np.nanmean(np.hypot(ex - ref_x, ey - ref_y), axis=1)
    rad_t = np.nanmean(er, axis=1)
    cover = 1.0 - np.isnan(ex).mean(axis=1)          # per-trial bin coverage

    def z(v):
        v = np.asarray(v, float); m, s = np.nanmean(v), np.nanstd(v)
        return (v - m) / s if s > 0 else np.full_like(v, np.nan)

    return pd.DataFrame(dict(
        mouse=mouse, day=day, trial=trials.number.values,
        ttype=trials.type.values, perf=trials.performance.values,
        n_cells=L.shape[0], frac_anch=frac,
        rad_raw=rad_t, dev_raw=dev, z_rad=z(rad_t), z_dev=z(dev),
        coverage=cover, blink_frac=blink,
        n_spike=n_spike, n_step=n_step, hm_frac=hm_frac))
print("batch helpers ready")

In [ ]:
# Set ONLY to e.g. 'M22D35' to run one session for a quick check. A partial run
# writes to its own file: an earlier single-session run overwrote the full
# 80-session CSV, which is silent and costs 15 minutes to undo.
ONLY = None

ds = sorted(glob.glob(f'{SOURCE_PATH}M*/D*/VR'))
out, t0 = [], time.time()
for i, d in enumerate(ds, 1):
    m = re.search(r'M(\d+)/D(\d+)/VR', d); mo, dy = int(m.group(1)), int(m.group(2))
    if ONLY and f'M{mo}D{dy}' != ONLY:
        continue
    bp, cp = vr_paths(mo, dy)
    if not (os.path.exists(bp) and os.path.exists(cp)):
        continue
    try:
        r = anchoring_and_eye(mo, dy)
    except Exception as e:
        print(f'  ! M{mo}D{dy}: {type(e).__name__}: {e}', flush=True); continue
    if r is None:
        continue
    out.append(r)
    print(f'[{i}/{len(ds)}] M{mo}D{dy}: {len(r)} trials, {r.n_cells.iloc[0]} cells, '
          f'blink {100*r.blink_frac.iloc[0]:.1f}%, cover {r.coverage.mean():.2f}, '
          f'slips {int(r.n_step.iloc[0])}, spikes {int(r.n_spike.iloc[0])}, '
          f'{time.time()-t0:.0f}s', flush=True)

D = pd.concat(out, ignore_index=True)
path = TRIALS_CSV if ONLY is None else TRIALS_CSV.replace('.csv', f'_{ONLY}.csv')
D.to_csv(path, index=False)
print(f'\n{len(D)} trials from {D.groupby(["mouse","day"]).ngroups} sessions -> {path}')
if ONLY:
    print('PARTIAL RUN -- the full CSV was left untouched.')


## Session inclusion criteria

In [ ]:
"""Session inclusion, defined ONCE and used by every analysis below.

Deliberately permissive: only trials whose measure is undefined and sessions
where tracking plainly failed are dropped. Tighten here -- nothing downstream
hard-codes these numbers.
"""
MIN_COVERAGE   = 0.50    # per-trial fraction of position bins with a camera sample
MAX_BLINK      = 0.20    # per-session fraction of frames failing DLC likelihood
MIN_TRIALS     = 40      # per-session total, after the above
MIN_RAD_CV     = 0.015   # reject sessions where the pupil never moves: tracking stuck.
# Calibrated, not guessed: per-session CV of trial-mean radius is unimodal over
# [0.029, 0.070] with no low outliers, so nothing is actually stuck and this
# criterion currently excludes nothing. An absolute SD threshold does NOT work --
# SD of the trial MEAN is ~0.3-0.7 px, so a 0.5 px cut removes half the sessions
# by slicing a unimodal distribution down the middle.
MIN_PER_CLASS  = 30      # trials in EACH of the anchored / non-anchored classes.
# Calibrated by subsampling the 24 sessions with >=90 trials in both classes down
# to n per class and measuring the spread of the estimate around that session's
# own full-data value. The estimator is unbiased at every n (|bias| <= 0.01), so
# this is purely about precision:
#     n/class      5     10     15     20     25     30     40     50
#     SD        0.61   0.42   0.34   0.28   0.25   0.22   0.18   0.16
# The effect is ~0.55 z. At n=20 the 95% error band is +-0.55, as wide as the
# effect, so such a session cannot establish its own sign. At n=30 the SD is
# ~half the effect, where a per-session value becomes interpretable; n=40 buys
# little for 5 more sessions lost. The pooled result does not depend on this
# (median -0.561 at no threshold vs -0.549 at 30, p <= 3e-10 throughout).


def load_trials(verbose=True):
    """Per-trial table with inclusion applied. Returns (included, session_table)."""
    D = pd.read_csv(TRIALS_CSV)
    D['ok_trial'] = (D.coverage >= MIN_COVERAGE) & D.z_rad.notna() & D.z_dev.notna()
    D['anch'] = D.frac_anch > 0.5
    cls = (D[D.ok_trial].groupby(['mouse', 'day']).anch
             .agg(n_anch='sum', n_tot='size').reset_index())
    cls['minority'] = np.minimum(cls.n_anch, cls.n_tot - cls.n_anch)
    sess = D.groupby(['mouse', 'day']).agg(
        blink=('blink_frac', 'first'), rad_sd=('rad_raw', 'std'),
        rad_med=('rad_raw', 'median'), n_ok=('ok_trial', 'sum'),
        n=('trial', 'size')).reset_index().merge(
            cls[['mouse', 'day', 'minority']], on=['mouse', 'day'], how='left')
    sess['minority'] = sess.minority.fillna(0)
    sess['rad_cv'] = sess.rad_sd / sess.rad_med
    sess['include'] = ((sess.blink <= MAX_BLINK) & (sess.n_ok >= MIN_TRIALS) &
                       (sess.rad_cv >= MIN_RAD_CV) & (sess.minority >= MIN_PER_CLASS))
    if verbose:
        print(f'{len(sess)} sessions, {int(sess.include.sum())} pass inclusion')
        for _, r in sess[~sess.include].iterrows():
            why = [n for n, c in (('blink', r.blink > MAX_BLINK),
                                  ('n_trials', r.n_ok < MIN_TRIALS),
                                  ('rad_cv', r.rad_cv < MIN_RAD_CV),
                                  ('per_class', r.minority < MIN_PER_CLASS)) if c]
            print(f'   excluded M{int(r.mouse)}D{int(r.day)}: {",".join(why)} '
                  f'(blink {r.blink:.3f}, n_ok {int(r.n_ok)}, '
                  f'rad_cv {r.rad_cv:.3f}, minority {int(r.minority)})')
    E = D.merge(sess[['mouse', 'day', 'include']], on=['mouse', 'day'])
    E = E[E.include & E.ok_trial].copy()
    if verbose:
        print(f'\n{len(E)} trials, {E.groupby(["mouse","day"]).ngroups} sessions, '
              f'{100*E.anch.mean():.1f}% anchored')
    return E, sess


print('inclusion criteria and load_trials() ready')

## Around the transitions

The pooled comparison treats every trial as a sample from one of two states.
Aligning to the moment the state *changes* is a stronger test: a difference that
appears as a step at the transition is harder to explain by a session-long
confound than one that only shows up in a pooled average.

**The window has to be state-pure, and this matters more than it sounds.** An
earlier version required only a run of 3 trials either side. Outside that the
state flickers badly — at lag −8, 46% of trials were already in the state being
entered, and only 15% of windows were clean across ±8. That contamination drags
the pre-transition plateau toward the post-transition value, which shrinks the
apparent total change and *inflates* the fraction of it that looks instantaneous.
Entry into the non-anchored state appeared to be a 95% one-trial step; on
state-pure windows it is 31% of a change twice as large.

So the window below requires the state to be constant across the whole of −4…−1
and constant across the whole of 0…+3. ±4 keeps ~196 transitions in 56 sessions;
±6 would leave ~80 in ~46.

In [ ]:
"""Pupil radius and gaze deviation around anchoring state transitions.

Trials are aligned to the FIRST trial of a new population state, separately for
entries into the anchored state and entries into the non-anchored state.

The window must be CLEAN: the state is required to be constant for the whole of
-WIN..-1 and constant for the whole of 0..WIN-1. This is stricter than requiring
a run of 3 either side, and the difference is not cosmetic. Under the weaker
3-trial rule the state outside +-3 flickers badly -- at lag -8, 46% of trials
were already in the state being entered, and only 15% of windows were clean
across +-8. That contamination pulls the pre-transition plateau toward the
post-transition value, shrinking the apparent total change and inflating the
fraction of it that appears to happen in one trial: entry into the non-anchored
state looked like a 95% one-trial step, when on clean windows it is 31% of a
change twice as large. The earlier reading -- an instantaneous pupil step and a
4-trial pupil lead -- was an artifact of window contamination, not biology.

WIN = 4 is the working compromise: it keeps ~196 transitions across 56 sessions.
WIN = 6 leaves only ~80 transitions in ~46 sessions.

Averaging is WITHIN session first and then across sessions. Pooling raw
transitions would weight sessions by how often they switched, and switch rate
varies several-fold between sessions.
"""
import os, warnings, numpy as np, pandas as pd
warnings.filterwarnings('ignore')
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon
plt.rcParams['font.family'] = 'Arial'

ANCH, NONANCH = '#9d5391', '#5f9ea0'
FIG = '/Users/harryclark/Documents/spatial-manifolds/scripts/figures/AnchorDynamics2026'
D = pd.read_csv('/Users/harryclark/Documents/spatial-manifolds/data/eye_anchoring/'
                'eye_anchoring_trials.csv')

MIN_COVERAGE, MAX_BLINK, MIN_TRIALS = 0.50, 0.20, 40
MIN_RAD_CV, MIN_PER_CLASS = 0.015, 30
WIN = 4          # trials either side; the state must be constant across each half

D['ok_trial'] = (D.coverage >= MIN_COVERAGE) & D.z_rad.notna() & D.z_dev.notna()
D['anch'] = D.frac_anch > 0.5
_ok = D[D.ok_trial]
_c = _ok.groupby(['mouse', 'day']).anch.agg(n_a='sum', n_t='size').reset_index()
_c['minority'] = np.minimum(_c.n_a, _c.n_t - _c.n_a)
sess = D.groupby(['mouse', 'day']).agg(
    blink=('blink_frac', 'first'), rad_sd=('rad_raw', 'std'),
    rad_med=('rad_raw', 'median'), n_ok=('ok_trial', 'sum')).reset_index().merge(
        _c[['mouse', 'day', 'minority']], on=['mouse', 'day'], how='left')
sess['minority'] = sess.minority.fillna(0)
sess['include'] = ((sess.blink <= MAX_BLINK) & (sess.n_ok >= MIN_TRIALS) &
                   (sess.rad_sd / sess.rad_med >= MIN_RAD_CV) &
                   (sess.minority >= MIN_PER_CLASS))
E = D.merge(sess[['mouse', 'day', 'include']], on=['mouse', 'day'])
E = E[E.include].copy()
print(f'{E.groupby(["mouse","day"]).ngroups} sessions')


def transitions(state, into):
    """First trial of each run of `into` whose +-WIN window is state-pure."""
    out = []
    for i in range(WIN, len(state) - WIN + 1):
        if state[i] != into or state[i - 1] == into:
            continue
        if (state[i - WIN:i] == into).any() or (state[i:i + WIN] != into).any():
            continue                      # flicker anywhere in the window
        out.append(i)
    return out


lags = np.arange(-WIN, WIN)
res = {}
for into, lab in ((True, 'anchored'), (False, 'non-anchored')):
    per_sess = {'z_rad': [], 'z_dev': []}
    n_tr = 0
    for _, g in E.groupby(['mouse', 'day']):
        g = g.sort_values('trial')
        st = g.anch.values
        idx = transitions(st, into)
        if not idx:
            continue
        for col in ('z_rad', 'z_dev'):
            v = g[col].values
            segs = []
            for i in idx:
                lo, hi = i - WIN, i + WIN
                if lo < 0 or hi > len(v):
                    continue
                segs.append(v[lo:hi])
            if segs:
                per_sess[col].append(np.nanmean(segs, axis=0))
        n_tr += len(idx)
    res[lab] = ({k: np.array(v) for k, v in per_sess.items()}, n_tr)
    print(f'  entering {lab:13s}: {n_tr} transitions in {len(per_sess["z_rad"])} sessions')

# ------------------------------------------------------------------ figure
fig, axes = plt.subplots(2, 2, figsize=(6.2, 5.4), sharex=True,
                         gridspec_kw=dict(hspace=.14, wspace=.26))
MEAS = (('z_rad', 'z(pupil radius)', 'k'),
        ('z_dev', 'z(gaze deviation)', 'k'))
for r, (col, ylab, cline) in enumerate(MEAS):
    for c, (into, lab) in enumerate(((True, 'anchored'), (False, 'non-anchored'))):
        ax = axes[r, c]
        M, n_tr = res[lab]
        A = M[col]
        m = np.nanmean(A, axis=0)
        se = np.nanstd(A, axis=0) / np.sqrt(np.sum(np.isfinite(A), axis=0))
        pre_c = NONANCH if into else ANCH          # state being left
        post_c = ANCH if into else NONANCH         # state being entered

        # highlight the two trial regions, drawn behind everything
        lo, hi = (-0.55, 0.62) if col == 'z_rad' else (-0.30, 0.42)
        ax.fill_between([-WIN, 0], lo, hi, color=pre_c, alpha=.13,
                        linewidth=0, edgecolor='none', zorder=0)
        ax.fill_between([0, WIN - 1], lo, hi, color=post_c, alpha=.13,
                        linewidth=0, edgecolor='none', zorder=0)
        ax.axvline(0, color='0.35', lw=1.1, ls='--', zorder=2)
        ax.axhline(0, color='0.75', lw=.8, zorder=1)

        ax.fill_between(lags, m - se, m + se, color=cline, alpha=.20,
                        linewidth=0, edgecolor='none', zorder=3)
        ax.plot(lags, m, color=cline, lw=1.5, zorder=4)
        ax.scatter(lags, m, s=11, color=cline, zorder=5)

        # paired test: mean of the WIN trials before vs after (both state-pure)
        pre = np.nanmean(A[:, :WIN], axis=1); post = np.nanmean(A[:, WIN:], axis=1)
        ok = np.isfinite(pre) & np.isfinite(post)
        p = wilcoxon(pre[ok], post[ok])[1]
        ax.set_ylim(lo, hi)
        if r == 0:
            ax.set_title(f'entering {lab}\n({n_tr} transitions, {ok.sum()} sessions)',
                         fontsize=8.5, pad=4)
        ax.text(.03, .04, f'Δ = {np.mean(post[ok]-pre[ok]):+.3f}\np = {p:.2g}',
                transform=ax.transAxes, fontsize=7, color='0.25', linespacing=1.3)
        if c == 0:
            ax.set_ylabel(ylab, fontsize=8.5)
        if r == 1:
            ax.set_xlabel('Trial rel. to transition', fontsize=8.5)
        ax.tick_params(labelsize=7.5)
        ax.spines[['top', 'right']].set_visible(False)
        if r == 0:
            ax.text(-WIN + .15, hi - .02, 'non-anch.' if into else 'anch.',
                    fontsize=6.5, color=pre_c, va='top', weight='bold')
            ax.text(.15, hi - .02, 'anch.' if into else 'non-anch.',
                    fontsize=6.5, color=post_c, va='top', weight='bold')

fig.suptitle('Pupil and gaze around anchoring transitions\n'
             '(state-pure windows)', fontsize=10, y=1.005)
p = f'{FIG}/eye_anchoring_transitions.pdf'
plt.savefig(p, dpi=200, bbox_inches='tight')
print('\nsaved', p)
plt.show()


## Around the transitions

The pooled comparison above treats every trial as a sample from one of two
states. This aligns trials to the moment the population state *changes*, which
is a stronger test: a difference that appears as a step at the transition is
harder to explain by a session-long confound than one that only shows up in a
pooled average.

A transition counts only if the old state held for 3 trials and the new state
holds for 3 — the classifier's median filter already erases runs shorter than
three, which sets the floor. Averaging is within session first, then across
sessions, because switch rate varies several-fold between sessions and pooling
raw transitions would weight sessions by how often they switched.

In [ ]:
"""Pupil radius and gaze deviation around anchoring state transitions.

Trials are aligned to the FIRST trial of a new population state, separately for
entries into the anchored state and entries into the non-anchored state. A
transition only counts if the previous state held for MIN_RUN trials and the new
one holds for MIN_RUN as well, so the window is not contaminated by flicker; the
classifier's median filter already erases runs shorter than three trials, which
sets the floor.

Averaging is done WITHIN session first and then across sessions. Pooling raw
transitions would weight sessions by how often they switched, and switch rate
varies several-fold between sessions.
"""

WIN     = 8      # trials shown either side of the transition
MIN_RUN = 3      # trials the old and new states must each hold

E, _ = load_trials(verbose=False)
print(f'{E.groupby(["mouse","day"]).ngroups} sessions')


def transitions(state, into):
    """Indices of the first trial of each qualifying run of `into`."""
    out = []
    n = len(state)
    for i in range(MIN_RUN, n - MIN_RUN + 1):
        if state[i] != into or state[i - 1] == into:
            continue
        if all(state[i - k] == (not into) for k in range(1, MIN_RUN + 1)) and \
           all(state[i + k] == into for k in range(MIN_RUN)):
            out.append(i)
    return out


lags = np.arange(-WIN, WIN)
res = {}
for into, lab in ((True, 'anchored'), (False, 'non-anchored')):
    per_sess = {'z_rad': [], 'z_dev': []}
    n_tr = 0
    for _, g in E.groupby(['mouse', 'day']):
        g = g.sort_values('trial')
        st = g.anch.values
        idx = transitions(st, into)
        if not idx:
            continue
        for col in ('z_rad', 'z_dev'):
            v = g[col].values
            segs = []
            for i in idx:
                lo, hi = i - WIN, i + WIN
                if lo < 0 or hi > len(v):
                    continue
                segs.append(v[lo:hi])
            if segs:
                per_sess[col].append(np.nanmean(segs, axis=0))
        n_tr += len(idx)
    res[lab] = ({k: np.array(v) for k, v in per_sess.items()}, n_tr)
    print(f'  entering {lab:13s}: {n_tr} transitions in {len(per_sess["z_rad"])} sessions')

# ------------------------------------------------------------------ figure
fig, axes = plt.subplots(2, 2, figsize=(9.5, 5.4), sharex=True,
                         gridspec_kw=dict(hspace=.14, wspace=.20))
MEAS = (('z_rad', 'z(pupil radius)', '#3b6ea5'),
        ('z_dev', 'z(gaze deviation)', '#c04744'))
for r, (col, ylab, cline) in enumerate(MEAS):
    for c, (into, lab) in enumerate(((True, 'anchored'), (False, 'non-anchored'))):
        ax = axes[r, c]
        M, n_tr = res[lab]
        A = M[col]
        m = np.nanmean(A, axis=0)
        se = np.nanstd(A, axis=0) / np.sqrt(np.sum(np.isfinite(A), axis=0))
        pre_c = NONANCH if into else ANCH          # state being left
        post_c = ANCH if into else NONANCH         # state being entered

        # highlight the two trial regions, drawn behind everything
        lo, hi = (-0.42, 0.78) if col == 'z_rad' else (-0.30, 0.62)
        ax.fill_between([-WIN, 0], lo, hi, color=pre_c, alpha=.13,
                        linewidth=0, edgecolor='none', zorder=0)
        ax.fill_between([0, WIN - 1], lo, hi, color=post_c, alpha=.13,
                        linewidth=0, edgecolor='none', zorder=0)
        ax.axvline(0, color='0.35', lw=1.1, ls='--', zorder=2)
        ax.axhline(0, color='0.75', lw=.8, zorder=1)

        ax.fill_between(lags, m - se, m + se, color=cline, alpha=.30,
                        linewidth=0, edgecolor='none', zorder=3)
        ax.plot(lags, m, color=cline, lw=1.8, zorder=4)
        ax.scatter(lags, m, s=13, color=cline, zorder=5)

        # paired test: mean of the WIN trials before vs after
        pre = np.nanmean(A[:, :WIN], axis=1); post = np.nanmean(A[:, WIN:], axis=1)
        ok = np.isfinite(pre) & np.isfinite(post)
        p = wilcoxon(pre[ok], post[ok])[1]
        ax.set_ylim(lo, hi)
        if r == 0:
            ax.set_title(f'entering {lab}\n({n_tr} transitions, {ok.sum()} sessions)',
                         fontsize=9.5, pad=5)
        ax.text(.03, .04, f'Δ = {np.mean(post[ok]-pre[ok]):+.3f}   p = {p:.2g}',
                transform=ax.transAxes, fontsize=8, color='0.25')
        if c == 0:
            ax.set_ylabel(ylab, fontsize=9)
        if r == 1:
            ax.set_xlabel('Trial relative to transition', fontsize=9)
        ax.tick_params(labelsize=8)
        ax.spines[['top', 'right']].set_visible(False)
        if r == 0:
            ax.text(-WIN + .3, hi - .03, 'non-anchored' if into else 'anchored',
                    fontsize=7.5, color=pre_c, va='top', weight='bold')
            ax.text(.3, hi - .03, 'anchored' if into else 'non-anchored',
                    fontsize=7.5, color=post_c, va='top', weight='bold')

fig.suptitle('Pupil and gaze around anchoring state transitions', fontsize=11.5, y=.99)
p = f'{FIG}/eye_anchoring_transitions.pdf'
plt.savefig(p, dpi=200, bbox_inches='tight')
print('\nsaved', p)
plt.show()


## Summary

Both measures are **lower on anchored trials**, across 57 sessions / 15,015 trials:

| measure | anchored | non-anchored | difference | sessions |
|---|---|---|---|---|
| z(pupil radius)   | −0.188 | +0.266 | **−0.454** | 52/57 negative |
| z(gaze deviation) | −0.140 | +0.198 | **−0.339** | 51/57 negative |

When the population is anchored the pupil is smaller and the gaze is closer to
the animal's own typical trajectory; when it is non-anchored the pupil is larger
and the gaze wanders.

**Controls, both passed:**

- *Circular-shift null* — anchoring comes in long runs and pupil size drifts over
  a session, so two slowly-varying series can correlate strongly with no
  trial-by-trial relationship. Shifting the anchoring sequence against the eye
  sequence preserves the autocorrelation of both and destroys only their
  alignment: observed −0.549 against a null of [−0.089, +0.160], **p = 0.001**.
- *Partial correlation* with trial index removed: ρ = −0.283 → −0.303 for radius,
  −0.215 → −0.201 for deviation. Unchanged, so not session drift.

**Head-movement artifacts were rare and did not drive this.** Median 0 slips per
session (none at all in 44 of 80), at most 2.24% of frames removed, and the
statistics are unchanged before and after removal — a null result about the
artifact, now checked rather than assumed.

**The conclusion does not depend on the inclusion thresholds.** Median effect
runs −0.561 (no per-class minimum) to −0.507 (≥50 per class), p ≤ 3×10⁻⁹
throughout. The per-class minimum governs which *individual* session values can
be read, not whether the pooled effect exists.

## Where it does not hold

**M25 is an outlier animal.** Median effect +0.040 against −0.32 to −1.16 for
every other mouse (Mann-Whitney p = 1.2×10⁻⁴); 3 of its 6 retained sessions run
positive, where every other animal has at most one. This is not a data-quality
artifact — M25's blink rate, coverage, pupil radius and head-slip counts are all
unremarkable. Within M25 the sign tracks the anchored fraction, which points at
the anchoring classifier behaving unusually in this animal rather than at the eye
measures. Worth checking whether M25 is also anomalous in the ADI and
stopping-strategy analyses before this figure is used.

The only session that significantly *reversed* on both measures, M25 D18, is
excluded by the per-class minimum (23 trials in its minority class).

## The confound that remains

The silent-trial guard forces trials with no spikes to non-anchored, and silence
is coordinated across cells. Trials where the animal stood still are therefore
enriched in the non-anchored class — and standing still also changes pupil size.
The circular-shift null does not address this, because it is a genuine
trial-level coupling, just not the one being claimed. The control is to restrict
to trials where the animal actually ran, which needs per-trial speed added to the
batch.


## Example figure: the eye, the population state, and pupil dilation

`fig5_pupil_arousal.py`, run below. The script is the single source — this cell
reproduces it rather than holding a copy, because the copy that used to live
here had drifted several revisions behind and would have overwritten the
current figure with an older one on any full run of this notebook.

**Row 1** — one anchored and one non-anchored trial, seven moments in a single
traversal each (0, 30, 60, 90, 120, 150, 170 cm) with the eight DeepLabCut
landmarks drawn on, beside the mean pupil radius against track position for the
two state blocks.

**Row 2** — the MEC population anchoring raster (cells in PC1 order, highest
loading at the left, as in `fig1_v2`), PC1 itself, and the same pupil radius as
a per-trial heatmap. All three share the trial axis; the heatmap also shares the
position axis with the curve above it. Dashed lines mark major state
transitions, so a block in the raster can be read straight across into the
pupil rows.

**Row 3** — the population result over all 23 included sessions, so the example
above is shown to be representative rather than asserted to be:

| | Δ z(pupil radius) | p |
|---|---|---|
| anchored vs non-anchored (paired, 23 sessions) | −0.670 | 2×10⁻⁶ |
| entering anchored (103 transitions, 22 sessions) | −0.407 | 8×10⁻⁵ |
| entering non-anchored (105 transitions, 22 sessions) | +0.388 | 5×10⁻⁵ |

The pupil is reliably **smaller** in the anchored state, and it moves at the
transition rather than drifting.

### The dilation difference is not localised on the track

The last panel was built to ask *where* on the track the states differ. The
answer is a negative: essentially everywhere, by a near-constant offset.

| track region | Δ (anchored − non-anchored) |
|---|---|
| 0–30 cm (black box in) | −0.380 |
| 30–90 cm | −0.197 |
| 90–110 cm (reward zone) | −0.248 |
| 110–170 cm | −0.247 |
| 170–200 cm (black box out) | −0.344 |

48 of 50 bins reach p < 0.05, the per-bin Δ spans only −0.43 to −0.14, and its
SD across the track is 0.075 against a mean of −0.266. The mild deepening in
the black boxes is the only structure and it is small. Note the per-bin tests
are uncorrected and neighbouring bins share traversals, so the markers show
extent rather than testing it.

Position profiles are rebuilt from the **gated** labels and cached to
`data/eye_anchoring/eye_position_profiles_gated.npz`.
`eye_anchoring_profiles.npz` is NOT used: it predates the absolute gate and
splits trials differently (M25D23 is 99/102 there against 113/88 now), so its
profiles describe a different partition of the data.

Frames are cached per session in `eye_frames_M*D*_trials.npz`, so the figure
rebuilds without `/Volumes/INCR-NolanLab` mounted. A session whose frames are
not cached needs the volume.

In [ ]:
"""The eye example figure, run from the script so the two cannot diverge.

`_g` keeps the script's namespace, so its helpers stay available below:
    _g['eye_trials']()          per-trial eye table, inclusion applied
    _g['position_profiles']()   per-session z(radius) against track position
    _g['transition_segments']() the state-transition windows

M29 D17 is attempted and skipped: under the gated classifier it has no
non-anchored trials at all (frac_anch 0.67 across 182 trials), so it cannot
serve as a two-state example. 25 sessions with eye tracking do have both
states with blocks of >= 15 trials -- M28 D25 (84/68) is the best-balanced
replacement -- but their frames are not cached, so building one needs the lab
volume mounted.
"""
import runpy
from IPython.display import display

FIGDIR = ('/Users/harryclark/Documents/spatial-manifolds/scripts/figures/'
          'AnchorDynamics2026')
_g = runpy.run_path(f'{FIGDIR}/fig5_pupil_arousal.py', run_name='__main__')
print('\nrebuilt fig5_pupil_arousal_M21D19.pdf')